# Activity 6 — Convert field observations into machine-learning training data

**Learning objective:** Understand how the Tuesday field observations connect to the Sentinel-2 predictor variables.

This activity follows the existing `Merged_CleanV3.geojson` → `training_array` workflow.

In [ ]:
from pystac_client import Client
from dask.distributed import Client as DaskClient
from odc.stac import load, configure_s3_access
import geopandas as gpd
import pandas as pd
import numpy as np
import xarray as xr
import folium
import matplotlib.pyplot as plt

import odc.geo.xr  # activates .odc accessor

## Prerequisite

Run Activity 5 first so that `median` exists in memory, or run the preparation cells from Activity 5 in this notebook.

The training layer must contain the numeric field:

`randomforest`

which stores the class ID used to train the classifier.

In [ ]:
TRAINING_PATH = "Training_Data/Merged_CleanV3.geojson"

gdf = gpd.read_file(TRAINING_PATH, bbox=tuple(bbox))

print("Training features:", len(gdf))
print("CRS:", gdf.crs)
print("\nClass counts:")
print(gdf["randomforest"].value_counts().sort_index())

In [ ]:
# Visualise the training locations by class.
gdf.explore(column="randomforest", legend=True)

## 1. Match the training-point CRS to the satellite raster

In [ ]:
training = gdf.to_crs(median.odc.geobox.crs)

print("Satellite CRS:", median.odc.geobox.crs)
print("Training CRS:", training.crs)

## 2. Extract Sentinel-2 values at each training point

In [ ]:
training_da = (
    training
    .assign(
        x=training.geometry.x,
        y=training.geometry.y
    )
    .to_xarray()
)

training_values = (
    median
    .sel(training_da[["x", "y"]], method="nearest")
    .squeeze()
    .compute()
    .to_pandas()
)

training_values.head()

## 3. Join class labels to satellite predictor values

In [ ]:
training_array = pd.concat(
    [training["randomforest"], training_values],
    axis=1
)

# Remove coordinate/helper columns if present.
training_array = training_array.drop(
    columns=["y", "x", "spatial_ref"],
    errors="ignore"
)

# Remove samples where satellite data were unavailable.
before = len(training_array)
training_array = training_array.dropna()
after = len(training_array)

print("Samples before removing NoData:", before)
print("Samples available for training:", after)

training_array.head()

## 4. Inspect what the computer will learn from

Each row is now one observation:

`class ID | red | green | blue | nir08 | swir16 | ndvi`

This is the bridge between **field observations** and **machine learning**.

In [ ]:
print("Columns used by the model:")
print(training_array.columns.tolist())

print("\nSamples per class:")
print(training_array["randomforest"].value_counts().sort_index())